In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import umap
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CDSView, GroupFilter
from bokeh.layouts import column
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10
import panel as pn
pn.extension()
%matplotlib inline

In [ ]:
### Penguins
directory = "penguins.csv"
flag = 'species'
rowid = 'rowid'


# ### Coronary Artery Disease
# directory = "CAD.csv"
# # flag = 'VHD'
# flag = 'Cath'
# rowid = None


# ### Breast Cancer
# from sklearn.datasets import load_breast_cancer
# data = load_breast_cancer()
# dat = data.data
# y = data.target
# feature_names = data.feature_names
# df_breast_cancer = pd.DataFrame(data=dat, columns=feature_names)
# df_breast_cancer['flag'] = y
# df_breast_cancer
# df = df_breast_cancer
# flag = 'flag'
# rowid = None

In [ ]:
df = pd.read_csv(directory)         # Comment this out when using the Breast Cancer dataset

color_classifier = df[flag]


df = df.dropna()
df[flag].value_counts()
if rowid is not None:
    df = df.drop(rowid, axis=1)

df_numeric = df.select_dtypes(include=["int64","float64"])

scaled_df_numeric = StandardScaler().fit_transform(df_numeric.values) # Mean centers and standardizes data according to the formula:


In [ ]:

# The ENTIRE UMAP algorithm in 2 lines

reducer = umap.UMAP()       # I chose not to alter parameters
embedding = reducer.fit_transform(scaled_df_numeric)
embedding.shape



# BOKEH Interactive Lasso Plotting

# 1) Create a master source with everything
data = {
'x':         embedding[:,0],
'y':         embedding[:,1],
'size':      [8]*len(embedding),
flag:   [str(s) for s in color_classifier], 
'rowid':     list(range(len(embedding)))
}
source = ColumnDataSource(data)


# 2) Build factors and palette for color coding.
factors = sorted(set(data[flag]))
n = len(factors)
key = min(max(n, 3), 10)        # Color codings have to be at least 3 for bokeh, and we want no more than 10 for ease on the eyes
base_palette = Category10[key]  
palette = base_palette[:n]  

# 3) Plot the source incorporating Bokeh's lasso feature.
p = figure(tools="lasso_select", width=600, height=600)

for sp, color in zip(factors, palette):
    view = CDSView(filter=GroupFilter(column_name=flag, group=sp))
    p.circle(
        'x', 'y',
        source=source,
        view=view,
        size='size',
        fill_color=color,
        line_color=None,
        legend_label=sp
    )

p.legend.title        = flag
p.legend.location     = "top_right"
p.legend.click_policy = "hide"
p.legend.background_fill_alpha = 0.8

# 4) Incorporate "onclick" interactive feature.
output = pn.pane.Str("")
selected_indices = []

def save_indices(event=None):
    global selected_indices
    # now these are global indices into `source.data`
    selected_indices = source.selected.indices
    output.object = f"Saved {len(selected_indices)} indices: {selected_indices}"

button = pn.widgets.Button(name="Save Selection")
button.on_click(save_indices)

pn.Column(p, button, output)


EBM Code. Uncomment and run ONLY after selecting a cluster.

In [ ]:
# # EBM Algorithm
# from interpret.glassbox import ExplainableBoostingClassifier
# from interpret import show

# from interpret import set_visualize_provider
# from interpret.provider import InlineProvider
# set_visualize_provider(InlineProvider())

# # data = df.drop(columns=flag)
# data = df.iloc[selected_indices].drop(columns=flag)

# X = pd.DataFrame(data=data)
# y = np.where(X.index.isin(selected_indices), 0, 1)


# # Split the data into training and test sets
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2
# )

# # Initialize and train the Explainable Boosting Machine (EBM)
# ebm = ExplainableBoostingClassifier()
# ebm.fit(X_train, y_train)

# # Make predictions and evaluate the model
# y_pred = ebm.predict(X_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

# show(ebm.explain_global())

# 3D Demo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import umap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CDSView, GroupFilter
from bokeh.layouts import column
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10
import panel as pn
pn.extension()

In [ ]:
%matplotlib widget
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.widgets import Slider

np.random.seed(5)

N = 100

datamat = np.ones((N, 4))
datamat[:, 1:] = np.random.uniform(-1,1, (N,3))

f = [0.2, -.5, .5, 0]

y = np.sign(np.matmul(datamat, f))

# Sample data
x1 = datamat[:,1]
x2 = datamat[:,2]
x3 = datamat[:,3]

# Create the figure + 3D axes
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(x1, x2, x3, c=y, cmap='bwr', depthshade=True)

# Make room for sliders
plt.subplots_adjust(left=0.1, bottom=0.25)

# Slider axes: [left, bottom, width, height]
ax_az = plt.axes([0.1, 0.15, 0.8, 0.03])
ax_el = plt.axes([0.1, 0.1, 0.8, 0.03])

# Sliders
s_az = Slider(ax_az, 'Azimuth',   0, 360, valinit=45)
s_el = Slider(ax_el, 'Elevation', 0,  90, valinit=30)

# Update function
def update(val):
    ax.view_init(elev=s_el.val, azim=s_az.val)
    fig.canvas.draw_idle()

s_az.on_changed(update)
s_el.on_changed(update)

plt.show()


In [ ]:
plt.figure(figsize=(3, 3))

plt.scatter(x1, x2, c=y, cmap='bwr')

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("2D Scatter of x1 vs x2")
plt.show()

In [ ]:
df = pd.DataFrame(data=datamat, columns = ['bias', 'x1', 'x2', 'x3'])
flag = 'flag'
df['flag'] = y

color_classifier = df[flag]

df_numeric = df.select_dtypes(include=["int64","float64"])

scaled_df_numeric = StandardScaler().fit_transform(df_numeric.values) # Mean centers and standardizes data according to the formula:

df_numeric

In [ ]:

# The ENTIRE UMAP algorithm in 2 lines

reducer = umap.UMAP()       # I chose not to alter parameters
embedding = reducer.fit_transform(scaled_df_numeric)
embedding.shape



# BOKEH Interactive Lasso Plotting

# 1) Create a master source with everything
data = {
'x':         embedding[:,0],
'y':         embedding[:,1],
'size':      [8]*len(embedding),
flag:   [str(s) for s in color_classifier], 
'rowid':     list(range(len(embedding)))
}
source = ColumnDataSource(data)


# 2) Build factors and palette for color coding.
factors = sorted(set(data[flag]))
n = len(factors)
key = min(max(n, 3), 10)        # Color codings have to be at least 3 for bokeh, and we want no more than 10 for ease on the eyes
base_palette = Category10[key]  
palette = base_palette[:n]  

# 3) Plot the source incorporating Bokeh's lasso feature.
p = figure(tools="lasso_select", width=600, height=600)

for sp, color in zip(factors, palette):
    view = CDSView(filter=GroupFilter(column_name=flag, group=sp))
    p.circle(
        'x', 'y',
        source=source,
        view=view,
        size='size',
        fill_color=color,
        line_color=None,
        legend_label=sp
    )

p.legend.title        = flag
p.legend.location     = "top_right"
p.legend.click_policy = "hide"
p.legend.background_fill_alpha = 0.8

# 4) Incorporate "onclick" interactive feature.
output = pn.pane.Str("")
selected_indices = []

def save_indices(event=None):
    global selected_indices
    # now these are global indices into `source.data`
    selected_indices = source.selected.indices
    output.object = f"Saved {len(selected_indices)} indices: {selected_indices}"

button = pn.widgets.Button(name="Save Selection")
button.on_click(save_indices)

pn.Column(p, button, output)


3D Demo EBM Code:

In [ ]:
# from interpret.glassbox import ExplainableBoostingClassifier
# from interpret import show

# from interpret import set_visualize_provider
# from interpret.provider import InlineProvider
# set_visualize_provider(InlineProvider())

# # data = df.drop(columns=flag)
# data = df.drop(columns=flag)

# X = pd.DataFrame(data=data)
# y = np.where(X.index.isin(selected_indices), 0, 1)


# # Split the data into training and test sets
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2
# )

# # Initialize and train the Explainable Boosting Machine (EBM)
# ebm = ExplainableBoostingClassifier()
# ebm.fit(X_train, y_train)

# # Make predictions and evaluate the model
# y_pred = ebm.predict(X_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

# show(ebm.explain_global())